In [16]:
%%writefile requirements.txt

streamlit==1.52.0
pyngrok==7.5.0
python-dotenv==1.2.1
langchain==0.2.16
langchain-core==0.2.41
langchain-community==0.2.11
langchain-text-splitters==0.2.4
langgraph==0.2.3
langchain-ollama==0.1.1
semantic-router==0.0.61
pyppeteer==2.0.0
nest-asyncio==1.6.0
praw==7.7.1
cohere==5.5.0
replicate==1.0.7

Overwriting requirements.txt


In [17]:
!pip install -r requirements.txt

In [18]:
from google.colab import userdata
import os

# ambil api token dari secret
api_token = userdata.get("api_token")

# masukkin api token ke env var
os.environ["REPLICATE_API_TOKEN"] = api_token


In [19]:
from langchain_community.llms import Replicate

# Get LLM
llm = Replicate(
    model="anthropic/claude-3.5-haiku"
)

In [20]:
output = llm.invoke("Negara mana yang menjadi juara piala dunia tahun 2022")
print (output)

Argentina menjadi juara Piala Dunia FIFA 2022 yang diselenggarakan di Qatar. Mereka mengalahkan Prancis dalam pertandingan final yang sangat dramatis melalui adu penalti dengan skor 4-2, setelah pertandingan berakhir imbang 3-3 pada perpanjangan waktu. Kemenangan ini merupakan gelar Piala Dunia ketiga bagi Argentina, yang dipimpin oleh kapten legendaris Lionel Messi, yang juga dinobatkan sebagai Pemain Terbaik Piala Dunia 2022.


In [21]:
## from langchain_core.tools import tool

def parse_input(input_str):
  parts = input_str.split(";")
  return dict(part.split("=") for part in parts)

@tool
def multiply(input: str):
  """Multiply two numbers.
  Input format: 'a=123;b=123'
  """
  input_dict = parse_input(input)
  a = float(input_dict["a"])
  b = float(input_dict["b"])
  hasil = a*b
  return hasil

In [22]:
## multiply.invoke("a=7;b=7")

49.0

In [32]:
from langchain.agents import agent_types, initialize_agent, create_structured_chat_agent, AgentType, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_community.llms import Replicate
from langchain_core.tools import tool
from langchain import hub

# Bikin tools
tools = [multiply]

# Bikin LLM
llm = Replicate(
    model="anthropic/claude-3.5-haiku"
)

# Bikin memory
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

# Bikin Prompt
system_message = """Kamu adalah agent tahu semua hal.

"""

# Bikin Agent
agent_executor = initialize_agent(
    llm=llm,
    tools=tools,
    agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
    memory=memory,
    verbose=True,
    agent_kwargs={"system_message": system_message},
)



In [33]:
output = agent_executor.invoke({"input": "Halo"})
output



> Entering new AgentExecutor chain...
```json
{
    "action": "Final Answer",
    "action_input": "Halo! Saya adalah agen yang siap membantu Anda. Ada yang bisa saya bantu?"
}
```

> Finished chain.


{'input': 'Halo',
 'chat_history': [HumanMessage(content='Halo'),
  AIMessage(content='Halo! Saya adalah agen yang siap membantu Anda. Ada yang bisa saya bantu?')],
 'output': 'Halo! Saya adalah agen yang siap membantu Anda. Ada yang bisa saya bantu?'}

In [34]:
%%writefile bot.py

from langchain.agents import agent_types, initialize_agent, create_structured_chat_agent, AgentType, AgentExecutor
from langchain.memory import ConversationBufferMemory
from langchain_community.llms import Replicate
from langchain_core.tools import tool
from langchain import hub

from dotenv import load_dotenv
import streamlit as st
import requests
import os
import json


def parse_input(input_str):
    parts = input_str.split(";")
    return dict(part.split("=") for part in parts)

@tool
def multiply(input: str) -> str:
    """
    Multiply two numbers.
    Input format: 'a=123;b=213'
    """
    try:
        # parts = input.strip().split(" and ")
        # a, b = int(parts[0]), int(parts[1])
        input_dict = parse_input(input)
        a = float(input_dict['a'])
        b = float(input_dict['b'])
        return str(a * b)
    except Exception as e:
        return f"Something went wrong with the tool: {e}"


@tool
def cat_fact(input):
    """Get unique and random drink fact"""
    try:
      response = requests.get("https://catfact.ninja/fact?max_length=200")

      return str(response.json()['fact'])
    except Exception as e:
      return f"Something went wrong with the tool: {e}"


@tool
def get_weather(input: str) -> str:
    """Get current weather for given latitude & longitude.

    Use a search tool to get the city coordinates first before using this tool to get accurate & updated weather..

    Input format: 'lat=-6.2;lon=106.8'
    """

    try:
      # parts = input.strip().split(" and ")
      # lat, lon = parts[0], parts[1]
      input_dict = parse_input(input)
      lat = float(input_dict['lat'])
      lon = float(input_dict['lon'])
      response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current_weather=true")
      result = str(response.json())
      return result
    except Exception as e:
      return f"Something went wrong with the tool: {e}"



def build_agent():
    ### Build agent dulu bos ku
    load_dotenv()
    # search_token = os.environ['SEARCH_TOKEN']

    llm = Replicate(model="anthropic/claude-3.5-haiku")


    system_message = """Ingin makan apa hari ini?."""

    memory = ConversationBufferMemory(
        memory_key="chat_history",
        return_messages=True
    )

    tools = [
      multiply,
      cat_fact,
      get_weather,
    ]

    # This is the correct conversational agent
    agent_executor = initialize_agent(
        llm=llm,
        tools=tools,
        agent=AgentType.CHAT_CONVERSATIONAL_REACT_DESCRIPTION,
        memory=memory,
        agent_kwargs={"system_message": system_message},
        verbose=True,
        max_iterations=10,
        handle_parsing_errors=True
    )

    return agent_executor

Overwriting bot.py


In [35]:
%%writefile app.py

# Import semuanya dulu
import streamlit as st
from bot import build_agent

# Judul
st.title("Bo Bots")

# Session state
if "agent" not in st.session_state:
  st.session_state.agent = build_agent()

if "messages" not in st.session_state:
  st.session_state.messages = []

agent = st.session_state.agent

# Tombol-tombol dan UI
reset_chat_button = st.button("mulai chat")
if reset_chat_button:
  st.session_state.messages = []
  st.session_state.agent = build_agent()
  # user_input = None
  # ai_output = None


user_input = st.chat_input()


for m in st.session_state.messages:
  with st.chat_message(m["role"]):
    st.markdown(m["content"], unsafe_allow_html=True)


if user_input is not None:
  # with st.chat_message("human"):
  st.session_state.messages.append({
    "role": "human",
    "content": user_input,
  })

  with st.chat_message("user"):
    st.markdown(user_input)


  with st.spinner("Thinking.."):
    ai_output = ""

    for step in agent.stream({"input": user_input}):
      if "actions" in step.keys():
        for action in step["actions"]:
          with st.chat_message("assistant"):
            tool_name = action.tool
            tool_input = action.tool_input

            tool_message = f"""
              <div style="border-left: 5px solid #4CAF50; padding:6px 10px; background-color: #f9f9f9; border-radius:4px; font-size:14px;">
                🛠️ <b>{tool_name}</b> <code>{tool_input}</code>
              </div>
            """
            st.session_state.messages.append({
              "role": "🛠️",
              "content": tool_message,
            })


            st.markdown(tool_message, unsafe_allow_html=True)


      if "output" in step.keys():
        ai_output = step["output"]


  with st.chat_message("assistant"):
    # with st.chat_message("assistant"):
    st.session_state.messages.append({
      "role": "assistant",
      "content": ai_output,
    })



    st.markdown(ai_output, unsafe_allow_html=True)


    # st.text(agent.memory.chat_memory.messages)

Overwriting app.py


In [36]:
from google.colab import userdata
import os

ngrok_token = userdata.get('ngrok_token')
api_token = userdata.get('api_token')

# Put token to env variable
os.environ["REPLICATE_API_TOKEN"] = api_token

with open(".env", "w") as f:
    f.write(f"REPLICATE_API_TOKEN={api_token}")

In [37]:
import subprocess
import os
import signal
from pyngrok import ngrok, conf
import time


conf.get_default().auth_token = ngrok_token

# Kill previous ngrok tunnels and streamlit
ngrok.kill()

# Kill any previous streamlit running on 8501
!fuser -k 8501/tcp

# Start streamlit
process = subprocess.Popen(["streamlit", "run", "app.py"])

# Wait for it to spin up
time.sleep(5)

# Start new ngrok tunnel
public_url = ngrok.connect(8501)
print(f"Streamlit app running at: {public_url}")

8501/tcp:             8564
Streamlit app running at: NgrokTunnel: "https://nonfermentative-nonreducibly-lenore.ngrok-free.dev" -> "http://localhost:8501"


In [38]:
import streamlit as st
import time

with st.spinner("Wait for it...", show_time=True):
    time.sleep(5)
st.success("Done!")
st.button("Rerun")

2025-12-06 10:39:18.149 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:18.150 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:18.151 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:18.154 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:18.655 Thread 'Thread-8': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:18.656 Thread 'Thread-8': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:18.657 Thread 'Thread-8': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-12-06 10:39:23.155 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode

False